In [ ]:
import pandas as pd
import numpy as np

# Load data
churn_data = pd.read_csv('churn_data_processed.csv')
competitor_threat_df = pd.read_csv('competitor_threat_scores.csv')

# SECTION 1: Create Category Threat Scores
competitor_categories = {
    'Minimalist': ['Skincare', 'Wellness'],
    'Nykaa': ['Skincare', 'Makeup', 'Haircare', 'Accessories', 'Wellness'],
    # ... full mapping
}

def get_category_threat(category, competitor_categories, competitor_threat_df):
    competitors_in_cat = [
        comp for comp, cats in competitor_categories.items() 
        if category in cats or 'All' in cats
    ]
    threat_scores = competitor_threat_df[
        competitor_threat_df['brand_name'].isin(competitors_in_cat)
    ]['expansion_threat'].values
    
    return threat_scores.mean() if len(threat_scores) > 0 else 35

churn_data['category_threat_score'] = churn_data['primary_category'].apply(
    lambda cat: get_category_threat(cat, competitor_categories, competitor_threat_df)
)

# SECTION 2: Add Individual Variance
np.random.seed(42)
churn_data['individual_competitor_threat'] = churn_data['category_threat_score'] + \
    np.random.normal(0, 8, len(churn_data))
churn_data['individual_competitor_threat'] = churn_data['individual_competitor_threat'].clip(0, 100)

print(f"✅ Threat Score Distribution:")
print(churn_data['individual_competitor_threat'].describe())

# SECTION 3: Create Threat Tiers
churn_data['category_threat_tier'] = pd.cut(
    churn_data['individual_competitor_threat'],
    bins=[0, 33, 66, 100],
    labels=['Low', 'Medium', 'High']
)

print(f"\n✅ Threat Tier Distribution:")
print(churn_data['category_threat_tier'].value_counts())

# SECTION 4: Feature Summary
features = churn_data[[
    'Churn_binary', 'segment', 'MonthlyCharges', 'tenure',
    'category_threat_score', 'individual_competitor_threat', 'category_threat_tier'
]].copy()

print(f"\n✅ Feature Engineering Complete")
print(f"Features shape: {features.shape}")

# Save
churn_data.to_csv('churn_data_with_features.csv', index=False)
print(f"✅ Saved to churn_data_with_features.csv")